# Week 1 — Exploratory Analysis (SupplyPrescript)

Goal: understand the real open shipment history **before** training the delay model.

Run `python week1/ingest_real_data.py` once (USAID SCMS by default), then open this notebook
(or run `python week1/explore_data.py` for the same charts without Jupyter).

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path("..").resolve()
df = pd.read_csv(ROOT / "data" / "shipments.csv")
df.head()

In [ ]:
print(df.shape)
print(df["actual_delay_days"].describe())
df.groupby("supplier")["actual_delay_days"].agg(["count", "mean", "median"]).round(2)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(df["actual_delay_days"], bins=40, color="#2f6f5e", edgecolor="white")
ax.axvline(3, color="#a4462f", linestyle="--", label="late threshold (3d)")
ax.set_title("Actual delay distribution")
ax.legend()
plt.show()

In [ ]:
(
    df.assign(season=df["is_peak_season"].map({True: "Peak", False: "Off-peak"}))
    .groupby(["origin_region", "season"])["actual_delay_days"]
    .mean()
    .unstack()
    .plot(kind="bar", title="Peak-season effect by region", ylabel="Mean delay (days)")
)
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## Takeaways for the model

1. **Supplier reliability differs** — some vendors (e.g. Delta Cove) pull the mean delay up.
2. **Peak season adds delay** — Nov/Dec shipments need the `is_peak_season` feature.
3. **Distance / lead time** are continuous drivers the regressor can use.
4. Next step: `python week1/train_model.py` (XGBoost classifier + regressor).